# 01 — Decoder / Decoder

**Chapter 19 — File 5 of 9 / 第19章 — 第5个文件（共9个）**

---

## Summary / 总结

This script demonstrates **Implementing the Decoder Layer**.

本脚本演示 **Implementing the Decoder Layer**。

---
## Step 1 — Step 1

In [ ]:
from tensorflow.keras.layers import Layer, Dropout
from multihead_attention import MultiHeadAttention
from positional_encoding import PositionEmbeddingFixedWeights
from encoder import AddNormalization, FeedForward

---
## Step 2 — Implementing the Decoder Layer

In [ ]:
class DecoderLayer(Layer):
    def __init__(self, h, d_k, d_v, d_model, d_ff, rate, **kwargs):
        super().__init__(**kwargs)
        self.multihead_attention1 = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout1 = Dropout(rate)
        self.add_norm1 = AddNormalization()
        self.multihead_attention2 = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout2 = Dropout(rate)
        self.add_norm2 = AddNormalization()
        self.feed_forward = FeedForward(d_ff, d_model)
        self.dropout3 = Dropout(rate)
        self.add_norm3 = AddNormalization()

    def call(self, x, encoder_output, lookahead_mask, padding_mask, training):

---
## Step 3 — Multi-head attention layer

In [ ]:
multihead_output1 = self.multihead_attention1(x, x, x, lookahead_mask)

---
## Step 4 — Expected output shape = (batch_size, sequence_length, d_model)
Add in a dropout layer

In [ ]:
multihead_output1 = self.dropout1(multihead_output1, training=training)

---
## Step 5 — Followed by an Add & Norm layer

In [ ]:
addnorm_output1 = self.add_norm1(x, multihead_output1)

---
## Step 6 — Expected output shape = (batch_size, sequence_length, d_model)
Followed by another multi-head attention layer

In [ ]:
multihead_output2 = self.multihead_attention2(addnorm_output1, encoder_output,
                                                      encoder_output, padding_mask)

---
## Step 7 — Add in another dropout layer

In [ ]:
multihead_output2 = self.dropout2(multihead_output2, training=training)

---
## Step 8 — Followed by another Add & Norm layer

In [ ]:
addnorm_output2 = self.add_norm1(addnorm_output1, multihead_output2)

---
## Step 9 — Followed by a fully connected layer

In [ ]:
feedforward_output = self.feed_forward(addnorm_output2)

---
## Step 10 — Expected output shape = (batch_size, sequence_length, d_model)
Add in another dropout layer

In [ ]:
feedforward_output = self.dropout3(feedforward_output, training=training)

---
## Step 11 — Followed by another Add & Norm layer

In [ ]:
return self.add_norm3(addnorm_output2, feedforward_output)

---
## Step 12 — Implementing the Decoder

In [ ]:
class Decoder(Layer):
    def __init__(self, vocab_size, sequence_length, h, d_k, d_v, d_model, d_ff, n, rate,
                       **kwargs):
        super().__init__(**kwargs)
        self.pos_encoding = PositionEmbeddingFixedWeights(sequence_length, vocab_size,
                                                          d_model)
        self.dropout = Dropout(rate)
        self.decoder_layer = [DecoderLayer(h, d_k, d_v, d_model, d_ff, rate)
                              for _ in range(n)]

    def call(self, output_target, encoder_output, lookahead_mask, padding_mask, training):

---
## Step 13 — Generate the positional encoding

In [ ]:
pos_encoding_output = self.pos_encoding(output_target)

---
## Step 14 — Expected output shape = (number of sentences, sequence_length, d_model)
Add in a dropout layer

In [ ]:
x = self.dropout(pos_encoding_output, training=training)

---
## Step 15 — Pass on the positional encoded values to each encoder layer

In [ ]:
for i, layer in enumerate(self.decoder_layer):
            x = layer(x, encoder_output, lookahead_mask, padding_mask, training)

        return x

---
## Learning Notes / 学习笔记

- **概念**: Implementing the Decoder Layer 是机器学习中的常用技术。  
  *Implementing the Decoder Layer is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Decoder / Decoder
# Complete Code / 完整代码
# ===============================

from tensorflow.keras.layers import Layer, Dropout
from multihead_attention import MultiHeadAttention
from positional_encoding import PositionEmbeddingFixedWeights
from encoder import AddNormalization, FeedForward

# Implementing the Decoder Layer
class DecoderLayer(Layer):
    def __init__(self, h, d_k, d_v, d_model, d_ff, rate, **kwargs):
        super().__init__(**kwargs)
        self.multihead_attention1 = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout1 = Dropout(rate)
        self.add_norm1 = AddNormalization()
        self.multihead_attention2 = MultiHeadAttention(h, d_k, d_v, d_model)
        self.dropout2 = Dropout(rate)
        self.add_norm2 = AddNormalization()
        self.feed_forward = FeedForward(d_ff, d_model)
        self.dropout3 = Dropout(rate)
        self.add_norm3 = AddNormalization()

    def call(self, x, encoder_output, lookahead_mask, padding_mask, training):
        # Multi-head attention layer
        multihead_output1 = self.multihead_attention1(x, x, x, lookahead_mask)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Add in a dropout layer
        multihead_output1 = self.dropout1(multihead_output1, training=training)

        # Followed by an Add & Norm layer
        addnorm_output1 = self.add_norm1(x, multihead_output1)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Followed by another multi-head attention layer
        multihead_output2 = self.multihead_attention2(addnorm_output1, encoder_output,
                                                      encoder_output, padding_mask)

        # Add in another dropout layer
        multihead_output2 = self.dropout2(multihead_output2, training=training)

        # Followed by another Add & Norm layer
        addnorm_output2 = self.add_norm1(addnorm_output1, multihead_output2)

        # Followed by a fully connected layer
        feedforward_output = self.feed_forward(addnorm_output2)
        # Expected output shape = (batch_size, sequence_length, d_model)

        # Add in another dropout layer
        feedforward_output = self.dropout3(feedforward_output, training=training)

        # Followed by another Add & Norm layer
        return self.add_norm3(addnorm_output2, feedforward_output)

# Implementing the Decoder
class Decoder(Layer):
    def __init__(self, vocab_size, sequence_length, h, d_k, d_v, d_model, d_ff, n, rate,
                       **kwargs):
        super().__init__(**kwargs)
        self.pos_encoding = PositionEmbeddingFixedWeights(sequence_length, vocab_size,
                                                          d_model)
        self.dropout = Dropout(rate)
        self.decoder_layer = [DecoderLayer(h, d_k, d_v, d_model, d_ff, rate)
                              for _ in range(n)]

    def call(self, output_target, encoder_output, lookahead_mask, padding_mask, training):
        # Generate the positional encoding
        pos_encoding_output = self.pos_encoding(output_target)
        # Expected output shape = (number of sentences, sequence_length, d_model)

        # Add in a dropout layer
        x = self.dropout(pos_encoding_output, training=training)

        # Pass on the positional encoded values to each encoder layer
        for i, layer in enumerate(self.decoder_layer):
            x = layer(x, encoder_output, lookahead_mask, padding_mask, training)

        return x

---

➡️ **Next / 下一步**: File 6 of 9